In [1]:
import requests
import csv

cities = [
    ('Ahmedabad', 23.03, 72.58),
    ('Surat', 21.17, 72.83),
    ('Rajkot', 22.30, 70.80),
    ('Bhuj', 23.25, 69.67),
    ('Vadodara', 22.31, 73.18)
]

url = 'https://api.open-meteo.com/v1/forecast'
rows = []

for city, lat, lon in cities:
    params = {
        'latitude': lat,
        'longitude': lon,
        'daily': 'temperature_2m_max,temperature_2m_min,precipitation_sum',
        'forecast_days': 7,
        'timezone': 'Asia/Kolkata'
    }
    resp = requests.get(url, params=params, timeout=10).json()
    daily = resp['daily']
    for i in range(len(daily['time'])):
        rows.append({
            'city': city,
            'date': daily['time'][i],
            'temp_max_c': daily['temperature_2m_max'][i],
            'temp_min_c': daily['temperature_2m_min'][i],
            'rain_mm': daily['precipitation_sum'][i]
        })

with open('gujarat_forecast.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['city', 'date', 'temp_max_c', 'temp_min_c', 'rain_mm'])
    writer.writeheader()
    writer.writerows(rows)

print('saved rows:', len(rows))

saved rows: 35


In [2]:
import requests
import json

target_cities = ['Gandhinagar', 'Anand', 'Mehsana']
geo_url = 'https://geocoding-api.open-meteo.com/v1/search'
forecast_url = 'https://api.open-meteo.com/v1/forecast'

outdoor_picks = {}

for cname in target_cities:
    geo_resp = requests.get(geo_url, params={'name': cname, 'count': 1}, timeout=10).json()
    match = geo_resp['results'][0]
    lat, lon = match['latitude'], match['longitude']

    fc_resp = requests.get(
        forecast_url,
        params={
            'latitude': lat,
            'longitude': lon,
            'daily': 'temperature_2m_max,temperature_2m_min,precipitation_sum',
            'forecast_days': 7,
            'timezone': 'Asia/Kolkata'
        },
        timeout=10
    ).json()

    daily = fc_resp['daily']
    candidates = []
    for i in range(len(daily['time'])):
        t_max = daily['temperature_2m_max'][i]
        rain = daily['precipitation_sum'][i]
        if t_max <= 36:
            candidates.append({
                'date': daily['time'][i],
                'temp_max_c': t_max,
                'rain_mm': rain
            })

    if candidates:
        best = min(candidates, key=lambda d: d['rain_mm'])
        outdoor_picks[cname] = best
    else:
        outdoor_picks[cname] = None

with open('outdoor_days.json', 'w', encoding='utf-8') as f:
    json.dump(outdoor_picks, f, indent=2)

print(outdoor_picks)

{'Gandhinagar': {'date': '2026-11-25', 'temp_max_c': 32.2, 'rain_mm': 0.0}, 'Anand': {'date': '2026-11-25', 'temp_max_c': 32.6, 'rain_mm': 0.0}, 'Mehsana': {'date': '2026-11-25', 'temp_max_c': 32.0, 'rain_mm': 0.0}}
